# 초보자용 테이블 생성/데이터 입력 노트북

이 노트북은 다음 순서로 사용합니다.
1. `TABLE_NAME`, `SCHEMA`를 직접 작성
2. 테이블 생성 셀 실행
3. `ROWS`에 데이터 작성
4. 데이터 입력/조회 셀 실행


# STS 문항 데이터

In [2]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

# DB 경로: 프로젝트 루트 app.db (필요 시 이 줄만 수정)
# DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\app.db")
DB_PATH = Path(r"C:\Users\kmj\project\__project_original\2.0-modular\app.db")
TABLE_NAME = 'parent_item'  # 원하는 테이블 이름으로 변경

# type: INTEGER(정수) / REAL(실수) / TEXT / BLOB(바이너리) 중 하나 사용
# constraints 예시: 'PRIMARY KEY AUTOINCREMENT', 'NOT NULL', 'DEFAULT 0'
SCHEMA = [
    {'name': 'test_id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 검사명
    {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 세부검사 조건
    {'name': 'item_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, # 검사 문항
    {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'} # 검사 메타데이터
]
PRIMARY_KEY = ['test_id', 'sub_test_json'] # 복합 pk 설정

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)

설정 확인 완료
DB_PATH = C:\Users\kmj\project\__project_original\2.0-modular\app.db
TABLE_NAME = parent_item
SCHEMA = [{'name': 'test_id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'sub_test_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'item_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'meta_json', 'type': 'TEXT', 'constraints': 'NOT NULL'}]
PRIMARY_KEY = ['test_id', 'sub_test_json']


In [4]:
# 2) 테이블 생성
column_defs = []
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').strip()
    part = f'"{c_name}" {c_type}'
    if c_constraints:
        part += f' {c_constraints}'
    column_defs.append(part)

if PRIMARY_KEY:
    pk_cols = ', '.join([f'"{c}"' for c in PRIMARY_KEY])
    column_defs.append(f'PRIMARY KEY ({pk_cols})')

create_sql = f'CREATE TABLE IF NOT EXISTS "{TABLE_NAME}" (' + ', '.join(column_defs) + ')'
print('실행 SQL:', create_sql)

conn = sqlite3.connect(DB_PATH)
try:
    conn.execute(create_sql)
    conn.commit()
    print(f'테이블 생성 완료: {TABLE_NAME}')
finally:
    conn.close()

실행 SQL: CREATE TABLE IF NOT EXISTS "parent_item" ("test_id" TEXT NOT NULL, "sub_test_json" TEXT NOT NULL, "item_json" TEXT NOT NULL, "meta_json" TEXT NOT NULL, PRIMARY KEY ("test_id", "sub_test_json"))
테이블 생성 완료: parent_item


## 영아용

In [5]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [1, 0, 0],
                'end_exclusive': [3, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            '1': '주변이 소란스러워도 하던 놀이를 계속한다.',
            '2': '기분이 나쁠 때보다는 좋을 때가 더 많다.',
            '3': '목욕할 때 물을 튀기거나 발로 차는 등 많이 움직인다.',
            '4': '새로운 옷(또는 신발)을 입지 않으려고 한다.',
            '5': '뜻대로 되지 않으면 쉽게 운다.',
            '6': '가끔 보는 지인이 집에 오면 낯을 가린다.',
            '7': '다른 아이가 울면 따라 운다.',
            '8': '주의력이 요구되는 활동(퍼즐이나 책 등)을 좋아한다.',
            '9': '도전적인 신체 활동을 좋아한다(예: 높은 곳에 기어오르기).',
            '10': '항상 미소 짓거나 웃는다.',
            '11': '컨디션이 좋지 않을 때 지나치게 칭얼거린다.',
            '12': '낯선 성인을 만났을 때 엄마에게 매달린다.',
            '13': '잠자고 일어날 때면 짜증을 내거나 운다.',
            '14': '하루 중 대부분의 시간을 기분 좋게 보낸다.',
            '15': '놀이터나 키즈카페에서 놀 때 행동이 재빠르다.',
            '16': '놀이할 때 잘 웃는다.',
            '17': '익숙하지 않은 상황을 피하려고 한다.',
            '18': '처음 보는 친구에게 다가가기 어려워한다.',
            '19': '다른 사람이 다치는 것을 보면 움츠러든다.',
            '20': '새로운 활동을 시작하는 것을 주저한다.',
            '21': '사물보다 사람에게 관심이 더 많다.',
            '22': '처음 먹어보는 음식은 먹지 않으려고 한다.',
            '23': '잠들기 전에 잠투정이 있다.',
            '24': '사람들의 외적인 변화에 관심을 보인다(예: 안경, 수염, 헤어스타일, 액세서리 등).',
            '25': '매일 밖에 나가서 놀자고 한다.',
            '26': '좋아하는 장난감을 한참을 가지고 논다.',
            '27': '동화책을 읽어주면 주의를 기울여 듣는다.',
            '28': '잠자리가 바뀌면 잠들기 힘들다.',
            '29': '“안 돼”라고 하면 하던 행동을 멈춘다.',
            '30': '유모차나 카시트에서 계속 움직인다.',
            '31': '잠깐 기다리라고 하면 참을 수 있다.',
            '32': '새로운 상황에 적응하는 데 시간이 오래 걸린다.',
            '33': '아파하는 사람을 보면 얼굴을 찡그린다.',
            '34': '별일 아닌 것에도 즐거워한다.',
            '35': '옷을 입힐 때 많이 움직인다.',
            '36': '자주 칭얼거리거나 짜증을 낸다.',
            '37': '양육자의 말을 귀 기울여 듣는다.',
            '38': '사람들의 표정을 빤히 살펴보는 편이다.',
            '39': '낯설거나 새로운 장소에 가면 불편해한다.',
            '40': '한 번 징징거림이 시작되면 오래간다.',
            '41': '가만히 앉아 있지 않고 계속 움직인다.',
            '42': '아는 사람을 만나면 웃으며 반긴다.',
        },
        'meta_json': {'name': 'STS 6요인 기질검사 영아용',
                      'item_count': 42
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건


In [6]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute(f'SELECT * FROM "{TABLE_NAME}" ORDER BY rowid DESC')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()

총 1건
('STS', '{"age_range": {"as_of_time": "00:00:00", "start_inclusive": [1, 0, 0], "end_exclusive": [3, 0, 0]}, "gender": ["male", "female"]}', '{"1": "주변이 소란스러워도 하던 놀이를 계속한다.", "2": "기분이 나쁠 때보다는 좋을 때가 더 많다.", "3": "목욕할 때 물을 튀기거나 발로 차는 등 많이 움직인다.", "4": "새로운 옷(또는 신발)을 입지 않으려고 한다.", "5": "뜻대로 되지 않으면 쉽게 운다.", "6": "가끔 보는 지인이 집에 오면 낯을 가린다.", "7": "다른 아이가 울면 따라 운다.", "8": "주의력이 요구되는 활동(퍼즐이나 책 등)을 좋아한다.", "9": "도전적인 신체 활동을 좋아한다(예: 높은 곳에 기어오르기).", "10": "항상 미소 짓거나 웃는다.", "11": "컨디션이 좋지 않을 때 지나치게 칭얼거린다.", "12": "낯선 성인을 만났을 때 엄마에게 매달린다.", "13": "잠자고 일어날 때면 짜증을 내거나 운다.", "14": "하루 중 대부분의 시간을 기분 좋게 보낸다.", "15": "놀이터나 키즈카페에서 놀 때 행동이 재빠르다.", "16": "놀이할 때 잘 웃는다.", "17": "익숙하지 않은 상황을 피하려고 한다.", "18": "처음 보는 친구에게 다가가기 어려워한다.", "19": "다른 사람이 다치는 것을 보면 움츠러든다.", "20": "새로운 활동을 시작하는 것을 주저한다.", "21": "사물보다 사람에게 관심이 더 많다.", "22": "처음 먹어보는 음식은 먹지 않으려고 한다.", "23": "잠들기 전에 잠투정이 있다.", "24": "사람들의 외적인 변화에 관심을 보인다(예: 안경, 수염, 헤어스타일, 액세서리 등).", "25": "매일 밖에 나가서 놀자고 한다.", "26": "좋아하는 장난감을 한참을 가지고 논다.", "27": "동화책

## 유아용

In [7]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [3, 0, 0],
                'end_exclusive': [7, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            '1': '의견이 다를 때 쉽게 조율이 가능하다.',
            '2': '놀이터나 키즈카페에서 놀 때 활동 반경이 넓다.',
            '3': '새로운 상황에 적응시키기가 힘들다.',
            '4': '친구나 형제가 혼나는 것을 보면 걱정한다.',
            '5': '자주 짜증을 낸다.',
            '6': '별일 아닌 것에도 즐거워한다.',
            '7': '조용히 앉아서 노는 것보다 뛰어노는 것을 좋아한다.',
            '8': '기분이 나쁠 때보다는 좋을 때가 더 많다.',
            '9': '양육자의 기분을 살피기 위해 얼굴을 쳐다본다.',
            '10': '원하는 것을 얻기 위해 참고 기다릴 수 있다.',
            '11': '사소한 일에도 기분이 나빠진다.',
            '12': '낯선 사람을 만나면 인사하기 힘들어한다.',
            '13': '집에서 침대나 소파 위를 뛰어다니며 논다.',
            '14': '다른 사람이 우는 것을 보면 슬퍼한다.',
            '15': '가끔 보는 지인이 집에 오면 수줍어하고 조용해진다.',
            '16': '아는 사람을 만나면 웃으며 좋아한다.',
            '17': '하고 싶은 걸 하지 못하면 심하게 화를 낸다.',
            '18': '자신이 경험한 일에 대해 불평을 자주 한다.',
            '19': '함께 길을 걸을 때면 앞으로 먼저 뛰어나간다.',
            '20': '다른 사람의 말투나 감정 변화를 쉽게 알아차린다.',
            '21': '도전적인 신체 활동을 좋아한다(예: 정글짐 등).',
            '22': '하루 중 대부분의 시간을 기분 좋게 보낸다.',
            '23': '하던 일을 끝까지 계속한다.',
            '24': '혼자서도 기분 좋게 잘 논다.',
            '25': '새로운 친구에게 쉽게 다가가지 못하고 수줍어한다.',
            '26': '조금 힘든 작업을 할 때도 포기하지 않는다.',
            '27': '게임이나 경쟁에서 지면 분을 삭이지 못한다.',
            '28': '양육자의 지시나 설명을 잘 따른다.',
            '29': '가만히 앉아 있지 않고 계속 움직인다.',
            '30': '항상 미소 짓거나 웃는다.',
            '31': '더 큰 보상을 위해 당장의 보상을 포기할 수 있다.',
            '32': '저녁이 되어도 에너지가 넘친다.',
            '33': '책이나 영화 속 인물에게 감정 이입을 잘한다.',
            '34': '놀이에서 생각한 대로 완성될 때까지 작업을 계속한다(예: 레고, 퍼즐).',
            '35': '새로운 활동을 시작하는 것을 주저한다.',
            '36': '실망하거나 실패했을 때 강하게 반응(울음 또는 불평)한다.',
            '37': '어떤 일이 끝나거나 시작될 때까지 참고 기다릴 수 있다.',
            '38': '새로운 친구들과 같이 놀기보다는 옆에서 지켜본다.',
            '39': 'TV나 길에서 불쌍한 동물을 보면 마음 아파한다.',
            '40': '컨디션이 좋지 않을 때 지나치게 징징댄다.',
            '41': '집 안에서든 집 밖에서든 주로 뛰어다닌다.',
            '42': '친구들과 놀 때 자주 소리 내어 웃는다.',
            '43': '낯설거나 새로운 장소에 가면 불편해한다.',
        },
        'meta_json': {'name': 'STS 6요인 기질검사 유아용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건


## 성인용

In [10]:
# 3) 스키마 기준 데이터 입력
import json

# 자동 증가 PK 컬럼은 입력 대상에서 자동 제외됩니다.
insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

# 아래 데이터는 반드시 insert_columns와 같은 키를 가져야 합니다.
# TEXT 컬럼은 dict/list를 넣으면 자동으로 JSON 문자열로 변환합니다.
ROWS = [
    {
        'test_id': 'STS',
        'sub_test_json': {
            'age_range': {
                'as_of_time': '00:00:00',
                'start_inclusive': [18, 0, 0],
                'end_exclusive': [100, 0, 0],
            },
            'gender': ['male', 'female'],
        },
        'item_json': {
            '1': '다른 사람의 속뜻이나 진짜 기분을 쉽게 파악할 수 있다.',
            '2': '기분이 나쁠 때보다는 좋을 때가 많다.',
            '3': '처음 만나는 사람에게 쉽게 다가가기 어렵다.',
            '4': '사소한 일에도 기분이 나빠진다.',
            '5': '별로 힘들이지 않고 하루 종일 활동할 수 있다.',
            '6': '내 삶에 만족한다.',
            '7': '다른 사람의 기분을 잘 알아챈다.',
            '8': '일을 마칠 때까지는 피곤한 줄 모른다.',
            '9': '분위기 파악을 잘하는 편이다.',
            '10': '쉽게 불안해진다.',
            '11': '하고 싶은 것이 내 뜻대로 되지 않는 경우 심하게 짜증이 난다.',
            '12': '첫 만남에서 자기소개하는 것이 어렵다.',
            '13': '여러 가지 일을 힘들이지 않고 해내곤 한다.',
            '14': '방해를 받더라도 내가 하던 일을 끝까지 해내는 편이다.',
            '15': '대부분의 일상이 즐겁다.',
            '16': '새로운 상황에 적응하는 것이 쉽지 않다.',
            '17': '실망하거나 실패하면 다른 일을 할 수 없을 정도로 기분이 나빠진다.',
            '18': '하루 중 대부분의 시간을 기분 좋게 보낸다.',
            '19': '바쁘게 생활하는 것이 즐겁다.',
            '20': '하기 싫은 일이라도 해야 한다면 끝까지 완수한다.',
            '21': '다른 사람의 입장이나 관점을 중요하게 여긴다.',
            '22': '새로운 사람들을 만나는 모임에 가는 것이 불편하다.',
            '23': '익숙하지 않은 모임에서 말을 하기보다는 지켜보는 편이다.',
            '24': '해야 할 일들은 미루지 않고 해내는 편이다.',
            '25': '하루에 3~4가지 이상의 일정을 잘 소화한다.',
            '26': '인생은 재미있는 일로 가득 차 있다.',
            '27': '직장 상사나 윗사람에게 싫은 소리를 들으면 화가 나서 아무것도 하고 싶지 않다.',
            '28': '계획(자신 또는 타인과의 약속)을 세우면 잘 지키는 편이다.',
            '29': '타인의 기분을 상하게 하지 않으려고 노력한다.',
            '30': '나는 행복하다.',
            '31': '눈치가 빠른 편이다.',
            '32': '과제나 일에 대한 집중력이 좋다.',
            '33': '아무것도 안 하고 가만히 있는 것은 시간 낭비라고 생각한다.',
            '34': '쉽게 우울해진다.',
            '35': '새로운 취미활동을 시도하는 것이 쉽지 않다.',
            '36': '조금 힘든 일이라도 나에게 의미가 있다면 끝까지 해낸다.',
            '37': '상대방의 기분에 대해 신경을 쓰는 편이다.',
            '38': '다른 사람들은 자주 나에게 좋은 일이 있냐고 묻는다.',
            '39': '어떤 문제를 해결하기 전까지 포기하지 않는 편이다.',
            '40': '새로운 시도가 부담스럽다(장소, 활동, 상황 등).',
            '41': '집에 있는 것보다 밖에 나가는 것을 좋아한다.',
            '42': '짜증이 나면 그 기분이 오래 지속된다.',
            '43': '말이나 행동을 하기 전에 주변 사람들이 어떤 반응을 보일지 생각해 본다.',
        },
        'meta_json': {'name': 'STS 6요인 기질검사 성인용',
                      'item_count': 43
        },
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'\"{c}\"' for c in insert_columns])
# insert_sql = f'INSERT INTO \"{TABLE_NAME}\" ({col_sql}) VALUES ({placeholders})'

# insert 후 데이터 수정하고 싶을 때
# 기존 데이터를 덮어쓰도록 ON CONFLICT 절 추가 (sqlite)
conflict_columns = ['test_id', 'sub_test_json'] # pk 컬럼
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) ON CONFLICT({conflict_sql}) DO UPDATE SET {update_sql}'
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'입력 완료: {len(values)}건')
finally:
    conn.close()

입력 대상 컬럼: ['test_id', 'sub_test_json', 'item_json', 'meta_json']
입력 완료: 1건
